In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

%cd /home/smalani/PartialObservations

cwd = os.getcwd()
print(cwd)

/home/smalani/PartialObservations
/home/smalani/PartialObservations


In [2]:
from config import config
from main.utils import Network, MLP
import torch

f = np.load("minmax/minmax.npz")

xmax = f['arr_0']
xmin = f['arr_1']
print(xmax)
print(xmin)

norm_func = lambda input, device: (input - torch.tensor(xmin).float().to(device)) / \
                        ((torch.tensor((xmax - xmin)).float().to(device)) + 1e-10)
inv_norm_func = lambda input, device: input * ((torch.tensor((xmax - xmin)).float().to(device)) + 1e-10) \
                                + torch.tensor(xmin).float().to(device)

# Create the network architecture

if config["MODEL"]["BOX"] == 'Black':
    # Create the network architecture
    mlp = MLP(6, config["MODEL"]["NUM_HIDDEN"], 6)
    
    class my_Network(Network):
        def __init__(self, network, train_size, xdim, norm_func=lambda input, device: input,
                        inv_norm_func=lambda input, device: input, init_available=True, device=None, 
                        tf_prop=1., integrator='RK4', add_par_num=0):
            super(my_Network, self).__init__(network, train_size, xdim, norm_func,
                        inv_norm_func, init_available, device, 
                        tf_prop, integrator, add_par_num)

            self.additional_pars = torch.nn.Parameter((torch.zeros(6)-1).to(self.device), requires_grad = True) 

        def output(self, x, par):

            # ANN_input = torch.cat((self.norm_func(x), par/20), dim=-1)
            ANN_input_out = self.norm_func(x)
            ANN_input = torch.clip(ANN_input_out, min=-1., max=2.)
            out = self.net(ANN_input)
            out = self.inv_norm_func(out) * (100 ** (self.additional_pars))

            return out

elif config["MODEL"]["BOX"] == 'Grey' or config["MODEL"]["BOX"] == 'Gray':
    
    # Create the network architecture
    mlp = MLP(6, config["MODEL"]["NUM_HIDDEN"], 3)
    
    if config["MODEL"]["Parameters"] == 'Trainable':
        class my_Network(Network):
            def __init__(self, network, train_size, xdim, norm_func=lambda input, device: input,
                            inv_norm_func=lambda input, device: input, init_available=True, device=None, 
                            tf_prop=1., integrator='RK4', add_par_num=2):
                super(my_Network, self).__init__(network, train_size, xdim, norm_func,
                            inv_norm_func, init_available, device, 
                            tf_prop, integrator, add_par_num)

                self.additional_pars = torch.nn.Parameter((torch.cat(((torch.tensor([0.4476, -0.4859, -0.6419])), 
                                                                        (torch.zeros(4) + 1)))).to(self.device), 
                                                            requires_grad = True)

            def output(self, x_input, par):
                ANN_input = self.norm_func(x_input)
                ANN_output = self.net(ANN_input)
                ANN_output = ANN_output * (100 ** (self.additional_pars[:3]))

                x, y, z, u, v, g = torch.unbind(x_input, dim=-1)

                u1_prime_x, u2_prime_y, u3_prime_z = torch.unbind(ANN_output, dim=-1)                

                omega = self.additional_pars[-4] * 10
                sigma = self.additional_pars[-3] * 10
                rho = self.additional_pars[-2] / 10
                eta = self.additional_pars[-1]

                alpha, uf, _, _, _, _, _, _, uc1_prime, uc2_prime, uc3_prime = datagen.par_fun()

                output = []

                output.append(-alpha * x + u1_prime_x - uc1_prime * x)
                output.append(-alpha * y + u2_prime_y - uc2_prime * y)
                output.append(-alpha * z + u3_prime_z - uc3_prime * z)
                output.append(alpha * (uf - u) - u1_prime_x)
                output.append(-alpha * v + omega * u1_prime_x - u2_prime_y - sigma * u3_prime_z)
                output.append(-alpha * g + rho * u2_prime_y + eta * u3_prime_z)

                out = torch.stack((output), dim=-1)
                return out
            def raw_output(self, x_input, par):

                ANN_input = self.norm_func(x_input)
                ANN_output = self.net(ANN_input) * (100 ** (self.additional_pars[:3]))

                return ANN_output
    elif config["MODEL"]["Parameters"] == 'Fixed':
        class my_Network(Network):
            def __init__(self, network, train_size, xdim, norm_func=lambda input, device: input,
                            inv_norm_func=lambda input, device: input, init_available=True, device=None, 
                            tf_prop=1., integrator='RK4', add_par_num=2):
                super(my_Network, self).__init__(network, train_size, xdim, norm_func,
                            inv_norm_func, init_available, device, 
                            tf_prop, integrator, add_par_num)

                # self.additional_pars = torch.nn.Parameter((torch.zeros(2)-0.5).to(self.device), 
                #                             requires_grad = True)
                self.additional_pars = torch.nn.Parameter((torch.tensor([0.4476, -0.4859, -0.6419])).to(self.device), 
                                                                requires_grad = True)

                

            def output(self, x_input, par):
                
                ANN_input = self.norm_func(x_input)
                ANN_output = self.net(ANN_input)
                ANN_output = ANN_output * (100 ** (self.additional_pars))

                x, y, z, u, v, g = torch.unbind(x_input, dim=-1)
                u1_prime_x, u2_prime_y, u3_prime_z = torch.unbind(ANN_output, dim=-1)

                alpha, uf, omega, sigma, rho, eta, phi1, phi2, uc1_prime, uc2_prime, uc3_prime = datagen.par_fun(D=1/7.3, sf=2.5)

                output = []

                # output.append(-alpha * x + u1_prime * x - uc1_prime * x)
                # output.append(torch.zeros(output[-1].shape).to(self.device))
                # output.append(-alpha * z + u3_prime * z - uc3_prime * z)
                # output.append(alpha * (uf - u) - u1_prime * x)
                # output.append(-alpha * v + omega * u1_prime * x - u2_prime * y - sigma * u3_prime * z)
                # output.append(-alpha * g + rho * u2_prime * y + eta * u3_prime * z)

                output.append(-alpha * x + u1_prime_x - uc1_prime * x)

                output.append(-alpha * y + u2_prime_y - uc2_prime * y)
                # output.append(torch.zeros(output[-1].shape).to(self.device))

                output.append(-alpha * z + u3_prime_z - uc3_prime * z)
                output.append(alpha * (uf - u) - u1_prime_x)
                output.append(-alpha * v + omega * u1_prime_x - u2_prime_y - sigma * u3_prime_z)
                output.append(-alpha * g + rho * u2_prime_y + eta * u3_prime_z)

                out = torch.stack((output), dim=-1)
                return out
            def raw_output(self, x_input, par):

                ANN_input = self.norm_func(x_input)
                ANN_output = self.net(ANN_input) * (100 ** (self.additional_pars))

                return ANN_output

    else:
        raise ValueError("Tell me whether to train the parameters!")
else:
    raise ValueError("Tell me what box to use!")


network = my_Network(mlp, config["DATA"]["N_TRAIN"], 6, norm_func=norm_func, inv_norm_func=inv_norm_func, 
                      init_available=config["DATA"]["INIT_AVAILABLE"], integrator='RK4')
device = 'cpu'

filename = 'data/' + 'model_' + '.net'

print(filename)


state_dict = torch.load(filename, map_location=torch.device(device))
network.load_state_dict(state_dict, strict=False)

print(network)
network.double()

[7.76480999e+01 6.13450077e-02 3.52889567e-01 2.08507400e+02
 2.11354658e+03 8.21362308e-01]
[1.45181035e+01 1.12344629e-04 2.81247634e-01 3.24970859e+01
 3.97881635e+02 6.58996734e-01]
data/model_.net
my_Network(
  (net): MLP(
    (layers): ModuleList(
      (0): Linear(in_features=6, out_features=32, bias=True)
      (1): Linear(in_features=32, out_features=32, bias=True)
    )
    (activation): SiLU()
    (output_layer): Linear(in_features=32, out_features=3, bias=True)
  )
)


my_Network(
  (net): MLP(
    (layers): ModuleList(
      (0): Linear(in_features=6, out_features=32, bias=True)
      (1): Linear(in_features=32, out_features=32, bias=True)
    )
    (activation): SiLU()
    (output_layer): Linear(in_features=32, out_features=3, bias=True)
  )
)

In [3]:
f = np.load('minmax/limitcycle_20.npz')
myvar0 = f['arr_0']


f = np.load('minmax/limitcycle_detail.npz')
myvar0_detail = f['arr_0']

print(myvar0.shape)

(6, 20)


In [4]:
from sympy.utilities.iterables import multiset_permutations
import numpy as np

import itertools
x = np.linspace(0.98, 1.02, 10)
# x = np.linspace(1., 1.02, 1)
print(x)

p_arr = np.ones((6, 6*x.size))
print(p_arr.shape)

for j in range(x.size):
    for i in range(6):
        p_arr[i,6*j+i] = x[j]

parr = np.tile(p_arr,(myvar0.shape[1]))
myvar0arr = np.repeat(myvar0,(p_arr.shape[1]), axis=1)
RHS_Eval_vals = (parr * myvar0arr).T

print(RHS_Eval_vals.shape)

[0.98       0.98444444 0.98888889 0.99333333 0.99777778 1.00222222
 1.00666667 1.01111111 1.01555556 1.02      ]
(6, 60)
(1200, 6)


In [5]:
# data_train = datagen.generate_data(n_train=config["DATA"]["N_TRAIN"],
#                                    tmax=config["DATA"]["TMAX"],
#                                    x_sample_num=(config["DATA"]["X_SAMPLE_NUM"]-1)*10+1,
#                                    detail=True,
#                                    theta_random=False)

# RHS_Eval_vals = data_train[3]

In [6]:
from sympy.utilities.iterables import multiset_permutations
import numpy as np

import itertools
x = np.linspace(0.99, 1.01, 5)
p = itertools.product(x, repeat=6)

p_arr = np.array(list(p)).T

parr = np.tile(p_arr,(myvar0.shape[1]))
myvar0arr = np.repeat(myvar0,(p_arr.shape[1]), axis=1)
print(parr.shape)
print(myvar0arr.shape)

RHS_Eval_vals = (parr * myvar0arr).T
print(RHS_Eval_vals.shape)
print(myvar0arr[:,:2])
print(parr[:,:10])
print(RHS_Eval_vals[:4,:].T)
# print(myvar0arr)

(6, 312500)
(6, 312500)
(312500, 6)
[[5.00000000e+01 5.00000000e+01]
 [9.47035345e-28 9.47035345e-28]
 [2.85187797e-01 2.85187797e-01]
 [1.20150526e+02 1.20150526e+02]
 [1.25440965e+03 1.25440965e+03]
 [6.62657291e-01 6.62657291e-01]]
[[0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99 ]
 [0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99 ]
 [0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99 ]
 [0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99  0.99 ]
 [0.99  0.99  0.99  0.99  0.99  0.995 0.995 0.995 0.995 0.995]
 [0.99  0.995 1.    1.005 1.01  0.99  0.995 1.    1.005 1.01 ]]
[[4.95000000e+01 4.95000000e+01 4.95000000e+01 4.95000000e+01]
 [9.37564991e-28 9.37564991e-28 9.37564991e-28 9.37564991e-28]
 [2.82335919e-01 2.82335919e-01 2.82335919e-01 2.82335919e-01]
 [1.18949021e+02 1.18949021e+02 1.18949021e+02 1.18949021e+02]
 [1.24186555e+03 1.24186555e+03 1.24186555e+03 1.24186555e+03]
 [6.56030719e-01 6.59344005e-01 6.62657291e-01 6.65970578e-01]]


In [7]:
from BandFModel import datagen

print(RHS_Eval_vals.shape)
RHS_Eval_vals = RHS_Eval_vals.reshape((-1,6))
print(RHS_Eval_vals.shape)

ANN_output = network.output(torch.tensor(RHS_Eval_vals).reshape((-1,6)).to(network.device),
                             torch.tensor([6]).reshape((-1,1)).to(network.device)).cpu().detach().numpy()

pars = datagen.par_fun()
output = datagen.ode_fun(0., RHS_Eval_vals.T, pars)
print(ANN_output.shape)
print(output.shape)

(312500, 6)
(312500, 6)
(312500, 6)
(312500, 6)


In [8]:
from torch.autograd.functional import jacobian


def torchfun(x):
    return datagen.torch_ode_fun(0., x.T, pars).squeeze()

def batch_jacobian(f, x):
    f_sum = lambda x: torch.sum(f(x), axis=0)
    return jacobian(f_sum, x).permute(1,0,2)

def torchfun_pred(x):
    return network.output(x.reshape((-1,6)).to(network.device),
                             torch.tensor([6]).reshape((-1,1)).to(network.device)).cpu()

print(torch.tensor(RHS_Eval_vals.T).shape)

# jactrue = jacobian(torchfun, torch.tensor(RHS_Eval_vals)[[0],:].T)
# jacpred = jacobian(torchfun_pred, torch.tensor(RHS_Eval_vals)[[0],:])

xmax_arr = np.repeat(xmax.reshape(-1,1),6,axis=1)
jac_norm = xmax_arr / xmax_arr.T
# jac_norm = np.repeat(jac_norm.reshape(1,6,6), RHS_Eval_vals.shape[0], axis=0)

jactrue = batch_jacobian(torchfun, torch.tensor(RHS_Eval_vals))
jacpred = batch_jacobian(torchfun_pred, torch.tensor(RHS_Eval_vals))

# jac_frobnorm = np.sum(((jactrue - jacpred).numpy() / jac_norm) ** 2) / RHS_Eval_vals.shape[0]
# print(jac_frobnorm)

jac_frobnorm_L1 = np.sqrt(np.sum((np.sum(np.abs((jactrue - jacpred).numpy()), axis=0) / (jac_norm * RHS_Eval_vals.shape[0])) ** 2))
jac_frobnorm_L2 = np.sqrt(np.sum((np.sqrt(np.sum((jactrue - jacpred).numpy() ** 2, axis=0)) / (jac_norm * RHS_Eval_vals.shape[0])) ** 2))
jac_frobnorm_Linf = np.sqrt(np.sum((np.max(np.abs((jactrue - jacpred).numpy()), axis=0) / jac_norm) ** 2))
print(jac_frobnorm_L1)
print(jac_frobnorm_L2)
print(jac_frobnorm_Linf)


torch.Size([6, 312500])
1.1641938552013686
0.002105678726768393
1.4809991436020502


In [9]:
print(jactrue.shape)
print(jactrue.reshape(jactrue.shape[0],-1).shape)

torch.Size([312500, 6, 6])
torch.Size([312500, 36])


In [10]:
RHS_Tensor = torch.tensor(RHS_Eval_vals)
print(RHS_Tensor.shape)

unboundRHS = RHS_Tensor.unbind(dim=0)

def jacobian_in_batch(func, x):
    '''
    Compute the Jacobian matrix in batch form.
    Return (B, D_y, D_x)
    '''
    y = func(x)

    batch = y.shape[0]
    single_y_size = np.prod(y.shape[1:])
    y = y.view(batch, -1)
    vector = torch.ones(batch).to(y)

    # Compute Jacobian row by row.
    # dy_i / dx -> dy / dx
    # (B, D) -> (B, 1, D) -> (B, D, D)
    jac = [torch.autograd.grad(y[:, i], x, 
                               grad_outputs=vector, 
                               retain_graph=True,
                               create_graph=True)[0].view(batch, -1)
                for i in range(single_y_size)]
    jac = torch.stack(jac, dim=1)
    
    return jac

RHS = RHS_Eval_vals.copy()
RHS[RHS < 1e-20] = 0

x = torch.tensor(RHS.copy(), requires_grad=True)

jac_set = jacobian_in_batch(torchfun, x)

jactrue = batch_jacobian(torchfun, x)
jacpred = batch_jacobian(torchfun_pred, x)

torch.Size([312500, 6])


In [11]:
print(torch.max(x,dim=0))
print(torch.min(x,dim=0))

torch.return_types.max(
values=tensor([7.8732e+01, 0.0000e+00, 3.5661e-01, 2.1078e+02, 2.1398e+03, 8.3002e-01],
       dtype=torch.float64, grad_fn=<MaxBackward0>),
indices=tensor([ 59375,      0, 156750, 218850,  46895, 156254]))
torch.return_types.min(
values=tensor([1.4386e+01, 0.0000e+00, 2.7927e-01, 3.0709e+01, 3.9101e+02, 6.5071e-01],
       dtype=torch.float64, grad_fn=<MinBackward0>),
indices=tensor([218750,      0, 281250,  46875, 218750, 281250]))


In [12]:
print(jac_set.shape)
print(jactrue.shape)

print(jac_set[0,:,:])
print(jactrue[0,:,:])

torch.Size([312500, 6, 6])
torch.Size([312500, 6, 6])
tensor([[ 2.9722e-02,  0.0000e+00,  0.0000e+00,  2.0775e-03,  0.0000e+00,
         -1.7899e+01],
        [ 0.0000e+00, -2.5216e-02,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         -0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  1.4951e-03,  0.0000e+00,  6.6234e-07,
         -0.0000e+00],
        [-5.9882e-01,  0.0000e+00,  0.0000e+00, -2.0353e-01,  0.0000e+00,
          1.7899e+01],
        [ 5.8085e+00, -2.9388e-01, -3.6471e+00,  2.0152e-02, -2.0146e-01,
         -1.7362e+02],
        [ 0.0000e+00,  3.8612e-02,  4.7108e-01,  0.0000e+00,  8.5552e-07,
         -2.0145e-01]], dtype=torch.float64, grad_fn=<SliceBackward>)
tensor([[ 2.9722e-02,  0.0000e+00,  0.0000e+00,  2.0775e-03,  0.0000e+00,
         -1.7899e+01],
        [ 0.0000e+00, -2.5216e-02,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         -0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  1.4951e-03,  0.0000e+00,  6.6234e-07,
         -0.0000e+00],
        [-5.9882e-01,  0.0

In [16]:
jacpred_np = jacpred.clone().numpy()
jactrue_np = jactrue.clone().numpy()

print((np.max(jacpred_np,axis=0)) / 
        (np.max(jactrue_np,axis=0)))

[[-1.37136176e+01             inf             inf -2.88711181e+00
              inf  8.53509020e+00]
 [            inf  9.57285698e-01             inf             inf
             -inf            -inf]
 [           -inf             inf -1.03410935e+02            -inf
   2.40659757e+00            -inf]
 [-9.31908026e-03             inf            -inf  1.31686180e-01
              inf  3.42309011e+00]
 [ 1.96628494e-01 -1.08803153e+03 -2.38729771e+02 -2.87644175e+00
   7.17047771e-01  8.54646163e+00]
 [           -inf  2.69343050e+00  2.22899314e-01            -inf
   2.39524768e+00 -2.09290903e-01]]


/tmp/ipykernel_31871/1971636100.py:4: RuntimeWarning: divide by zero encountered in true_divide
  print((np.max(jacpred_np,axis=0)) /
